In [ ]:
date_tuple

In [ ]:
import os
import io
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from dateutil import parser, relativedelta
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline


# Pre-process for reading original data
rain_rate = []
num_particles = []
_base_time = []
nd = []
vd = []
raw = []
with io.open('../disdrodb-data/testdata.mis', encoding='utf-8') as f:
    for line in f:
        line = line.rstrip("\n\r;")
        code = line.split(":")[0]
        if code == "01":    # Rain Rate
            rain_rate.append(float(line.split(":")[1]))
        elif code == "11":  # Num Particles
            num_particles.append(int(line.split(":")[1]))
        elif code == "21":  # Date string
            date_tuple = line.split(":")[1].split(".")
            _base_time.append(
                datetime(
                    year=int(date_tuple[2]),
                    month=int(date_tuple[0]),
                    day=int(date_tuple[1]),
                )
            )
        elif code == "90":  # Nd
            nd.append(np.power(10, list(map(float, line.split(":")[1].split(";")))))
        elif code == "93":  # md
            raw.append(list(map(int, line.split(":")[1].split(";"))))

# Constants
diameter = np.array([0.06,0.19,0.32,0.45,0.58,
                     0.71,0.84,0.96,1.09,1.22,
                     1.42,1.67,1.93,2.19,2.45,
                     2.83,3.35,3.86,4.38,4.89,
                     5.66,6.7,7.72,8.76,9.78,
                     11.33,13.39,15.45,17.51,19.57,
                     22.15,25.24])
vel    = np.array([0.050,0.150,0.250,0.350,0.450,
                   0.550,0.650,0.750,0.850,0.950,
                   1.100,1.300,1.500,1.700,1.900,
                   2.200,2.600,3.000,3.400,3.800,
                   4.400,5.200,6.000,6.800,7.600,
                   8.800,10.400,12.000,13.600,15.200,
                   17.600,20.800])
spread = np.array([0.129,0.129,0.129,0.129,0.129,0.129,
                   0.129,0.129,0.129,0.129,0.257,0.257,
                   0.257,0.257,0.257,0.515,0.515,0.515,
                   0.515,0.515,1.030,1.030,1.030,1.030,
                   1.030,2.060,2.060,2.060,2.060,2.060,
                   3.090,3.090])


# Plot V-D histogram
plt.pcolormesh(np.reshape(raw, (32,32)), cmap='Blues')
plt.title('Diameter-Fall velocity')
plt.xlabel('Diameter in laser sheet pixels (mm)')
plt.ylabel('Fall velocity in laser sheet pixels (m s$^{-1}$)')
plt.xticks(np.arange(0,32,4), diameter[::4])
plt.yticks(np.arange(0,32,4), vel[::4])
plt.colorbar(extend='both', label='Number of counts')
plt.show()


# Plot drop size distribution
for k in range(len(raw)):
    raw_data = np.reshape(raw[k], (32,32))
    ND = np.array([0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0])
    for j in range(32): # V loop
        for i in range(32): # D loop
            ND[j] += raw_data[i][j] / (vel[i] * (0.18 * 0.03) * 60.0)  # md / (Vi * sheet area * sampling time)
        ND[j] /= spread[j]  # md / (Vi * sheet area * sampling time * delta Di)
    plt.plot(diameter, np.ma.masked_where(ND < 0.1, ND), linestyle=':')
plt.xlabel('Diameter (mm)')
plt.ylabel('$N(D)$ (mm$^{-1}$ m$^{-3}$)')
plt.yscale('log')
plt.ylim(1,10000)
plt.xlim(0,5)
plt.grid()
plt.show()